In [1]:
import os
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from tqdm import tqdm
import glob

In [5]:
import os
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from tqdm import tqdm

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5)

BASE_PATH = "asl_alphabet_train"
IMAGES_PER_CLASS = 1000

classes_to_use = [chr(i) for i in range(65, 91)] + ['space', 'nothing']

data = []

for label in classes_to_use:
    folder_path = os.path.join(BASE_PATH, label)
    if not os.path.isdir(folder_path):
        print(f"Warning: folder not found for {label}, skipping")
        continue

    img_list = os.listdir(folder_path)[:IMAGES_PER_CLASS]

    for img_name in tqdm(img_list, desc=label):
        img_path = os.path.join(folder_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(image_rgb)

        if results.multi_hand_landmarks:
            landmarks = results.multi_hand_landmarks[0]
            row = []
            for lm in landmarks.landmark:
                row.extend([lm.x, lm.y, lm.z])
            row.append(label)
            data.append(row)

columns = []
for i in range(21):
    columns.extend([f'x{i}', f'y{i}', f'z{i}'])
columns.append('label')

df = pd.DataFrame(data, columns=columns)
df.to_csv('train_landmarks.csv', index=False)
print(f"Done. Saved {len(df)} samples to train_landmarks.csv")

I0000 00:00:1784784683.858997   48944 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M5
W0000 00:00:1784784683.864928   56445 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1784784683.868438   56452 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
nothing: 100%|██████████| 1000/1000 [00:10<00:00, 95.03it/s]


Done. Saved 20745 samples to train_landmarks.csv


# Splitting and Model Training(Random Forest)

In [2]:
import pandas as pd

In [ ]:

from sklearn.model_selection import train_test_split

df= pd.read_csv('train_landmarks.csv')

X= df.drop(columns= 'label')
y= df['label']

#80% train+val, 20% test
X_temp, X_test, y_temp, y_test= train_test_split(X, y, test_size= 0.2, random_state=42, stratify= y)

#15% val_set, 65% train
X_train, X_val, y_train, y_val= train_test_split(X_temp, y_temp, test_size= 0.15, random_state= 42, stratify= y_temp)

print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

Train: 14106, Val: 2490, Test: 4149


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

val_preds = model.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print('Classification Repport:\n', classification_report(y_val, val_preds))

Validation Accuracy: 0.9686746987951808
Classification Repport:
               precision    recall  f1-score   support

           A       0.98      0.98      0.98        89
           B       1.00      0.99      0.99        88
           C       0.99      0.99      0.99        79
           D       1.00      0.97      0.99       102
           E       0.97      0.98      0.97        95
           F       0.98      0.98      0.98       113
           G       0.97      0.98      0.98       101
           H       0.99      0.98      0.98        94
           I       1.00      0.95      0.97        93
           J       0.99      0.97      0.98       104
           K       1.00      0.97      0.99       107
           L       0.99      0.99      0.99       101
           M       0.91      0.98      0.95        63
           N       1.00      0.94      0.97        49
           O       0.95      0.99      0.97        94
           P       0.99      0.94      0.96        84
           Q    

In [ ]:
train_preds = model.predict(X_train)
print("Train Accuracy:", accuracy_score(y_train, train_preds))

Train Accuracy: 1.0


#### -> Used for capturing photos

import cv2

cap = cv2.VideoCapture(0)
count = 0

while True:
    label = input("Enter the letter you're about to sign (or 'quit' to stop): ").upper()
    if label == 'QUIT':
        break

    print(f"Get ready to sign '{label}'... webcam window will open, press 's' to save, 'n' for next letter")
    while True:
        ret, frame = cap.read()
        cv2.imshow('Capture - press s to save, n for next letter', frame)
        key = cv2.waitKey(1)
        if key == ord('s'):
            filename = f"webcam_test_{label}_{count}.jpg"
            cv2.imwrite(filename, frame)
            print(f"Saved {filename}")
            count += 1
        elif key == ord('n'):
            break

cap.release()
cv2.destroyAllWindows()

In [16]:
import glob
import mediapipe as mp

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5)

test_images = glob.glob('webcam_test_*.jpg')
print(f"Found {len(test_images)} images:", test_images)

results_list = []

for img_path in test_images:
    true_label = img_path.split('_')[2]

    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = hands.process(image_rgb)

    if result.multi_hand_landmarks:
        landmarks = result.multi_hand_landmarks[0]
        row = []
        for lm in landmarks.landmark:
            row.extend([lm.x, lm.y, lm.z])
        
        pred = model.predict([row])[0]
        results_list.append((img_path, true_label, pred))
        print(f"{img_path}: True={true_label}, Predicted={pred}")
    else:
        print(f"{img_path}: No hand detected!")


I0000 00:00:1784880745.925673  370016 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M5
W0000 00:00:1784880745.930181  380985 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1784880745.934885  380985 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with

Found 26 images: ['webcam_test_H_16.jpg', 'webcam_test_N_6.jpg', 'webcam_test_N_4.jpg', 'webcam_test_N_5.jpg', 'webcam_test_H_15.jpg', 'webcam_test_M_7.jpg', 'webcam_test_N_12.jpg', 'webcam_test_M_1.jpg', 'webcam_test_M_3.jpg', 'webcam_test_N_10.jpg', 'webcam_test_N_11.jpg', 'webcam_test_M_2.jpg', 'webcam_test_A_2.jpg', 'webcam_test_A_3.jpg', 'webcam_test_G_13.jpg', 'webcam_test_A_1.jpg', 'webcam_test_A_0.jpg', 'webcam_test_M_9.jpg', 'webcam_test_A_4.jpg', 'webcam_test_A_5.jpg', 'webcam_test_M_8.jpg', 'webcam_test_A_6.jpg', 'webcam_test_G_14.jpg', 'webcam_test_B_5.jpg', 'webcam_test_B_4.jpg', 'webcam_test_B_6.jpg']
webcam_test_H_16.jpg: True=H, Predicted=C
webcam_test_N_6.jpg: True=N, Predicted=C
webcam_test_N_4.jpg: True=N, Predicted=C
webcam_test_N_5.jpg: True=N, Predicted=C
webcam_test_H_15.jpg: True=H, Predicted=C
webcam_test_M_7.jpg: True=M, Predicted=R
webcam_test_N_12.jpg: True=N, Predicted=C


/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

webcam_test_M_1.jpg: True=M, Predicted=C
webcam_test_M_3.jpg: True=M, Predicted=C
webcam_test_N_10.jpg: True=N, Predicted=C
webcam_test_N_11.jpg: True=N, Predicted=C
webcam_test_M_2.jpg: True=M, Predicted=C
webcam_test_A_2.jpg: True=A, Predicted=C
webcam_test_A_3.jpg: True=A, Predicted=C
webcam_test_G_13.jpg: True=G, Predicted=C
webcam_test_A_1.jpg: True=A, Predicted=W


/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

webcam_test_A_0.jpg: True=A, Predicted=C
webcam_test_M_9.jpg: True=M, Predicted=Z
webcam_test_A_4.jpg: True=A, Predicted=C
webcam_test_A_5.jpg: True=A, Predicted=C
webcam_test_M_8.jpg: True=M, Predicted=C
webcam_test_A_6.jpg: True=A, Predicted=C
webcam_test_G_14.jpg: True=G, Predicted=C
webcam_test_B_5.jpg: True=B, Predicted=C
webcam_test_B_4.jpg: True=B, Predicted=C
webcam_test_B_6.jpg: True=B, Predicted=C


/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


# Normalized Dataset and Training


In [4]:
import os
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from tqdm import tqdm

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5)

I0000 00:00:1785129985.750859  347542 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M5


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1785129985.756574  350167 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1785129985.759437  350166 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [13]:
import os
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from tqdm import tqdm

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5)

BASE_PATH = "asl_alphabet_train"
IMAGES_PER_CLASS = 1000

classes_to_use = [chr(i) for i in range(65, 91)] + ['space', 'nothing']

def normalize_landmarks(landmarks):
    coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks.landmark])

    # Step 1: make relative to wrist (landmark 0)
    wrist = coords[0].copy()
    coords -= wrist

    # Step 2: scale-normalize using wrist-to-middle-fingertip distance (landmark 12)
    scale = np.linalg.norm(coords[12])
    if scale > 0:
        coords /= scale

    return coords.flatten()  # 63 numbers, normalized

data = []

for label in classes_to_use:
    folder_path = os.path.join(BASE_PATH, label)
    if not os.path.isdir(folder_path):
        print(f"Warning: folder not found for {label}, skipping")
        continue

    img_list = os.listdir(folder_path)[:IMAGES_PER_CLASS]

    for img_name in tqdm(img_list, desc=label):
        img_path = os.path.join(folder_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(image_rgb)

        if results.multi_hand_landmarks:
            landmarks = results.multi_hand_landmarks[0]
            row = list(normalize_landmarks(landmarks))
            row.append(label)
            data.append(row)

columns = []
for i in range(21):
    columns.extend([f'x{i}', f'y{i}', f'z{i}'])
columns.append('label')

df = pd.DataFrame(data, columns=columns)
df.to_csv('train_landmarks_normalized.csv', index=False)
print(f"Done. Saved {len(df)} samples to train_landmarks_normalized.csv")

I0000 00:00:1784874003.088784  284363 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M5
W0000 00:00:1784874003.114087  304301 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
A:   0%|          | 0/1000 [00:00<?, ?it/s]W0000 00:00:1784874003.118367  304302 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
nothing: 100%|██████████| 1000/1000 [00:11<00:00, 88.44it/s]


Done. Saved 20745 samples to train_landmarks_normalized.csv


In [2]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv('train_landmarks_normalized.csv')
X = df.drop(columns='label')
y = df['label']

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Train: 14106, Val: 2490, Test: 4149


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

val_preds = model.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, val_preds))

Validation Accuracy: 0.9887550200803212


#### Capture ~30-50 images each for A, M, N, S

cap = cv2.VideoCapture(0)
count = 0

while True:
    label = input("Enter the letter you're about to sign (or 'quit' to stop): ").upper()
    if label == 'QUIT':
        break

    print(f"Signing '{label}' — press 's' to save, 'n' for next letter")
    while True:
        ret, frame = cap.read()
        cv2.imshow('Capture - press s to save, n for next letter', frame)
        key = cv2.waitKey(1)
        if key == ord('s'):
            filename = f"webcam_train_{label}_{count}.jpg"
            cv2.imwrite(filename, frame)
            print(f"Saved {filename}")
            count += 1
        elif key == ord('n'):
            break

cap.release()
cv2.destroyAllWindows()

In [10]:
def normalize_landmarks(landmarks):
    coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks.landmark])

    # Step 1: make relative to wrist (landmark 0)
    wrist = coords[0].copy()
    coords -= wrist

    # Step 2: scale-normalize using wrist-to-middle-fingertip distance (landmark 12)
    scale = np.linalg.norm(coords[12])
    if scale > 0:
        coords /= scale

    return coords.flatten()  # 63 numbers, normalized


In [11]:
import glob

new_data = []

train_images = glob.glob('webcam_train_*.jpg')

for img_path in train_images:
    true_label = img_path.split('_')[2]

    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = hands.process(image_rgb)

    if result.multi_hand_landmarks:
        landmarks = result.multi_hand_landmarks[0]
        row = list(normalize_landmarks(landmarks))
        row.append(true_label)
        new_data.append(row)

# Build dataframe with same columns as before
columns = []
for i in range(21):
    columns.extend([f'x{i}', f'y{i}', f'z{i}'])
columns.append('label')

new_df = pd.DataFrame(new_data, columns=columns)
print(f"Extracted {len(new_df)} new samples")

# Merge with existing training data
df_combined = pd.concat([df, new_df], ignore_index=True)
df_combined.to_csv('train_landmarks_augmented.csv', index=False)
print(f"Combined dataset: {len(df_combined)} samples")

/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Extracted 124 new samples
Combined dataset: 20869 samples


In [12]:
X_aug = df_combined.drop(columns='label')
y_aug = df_combined['label']

X_temp2, X_test2, y_temp2, y_test2 = train_test_split(X_aug, y_aug, test_size=0.2, random_state=42, stratify=y_aug)
X_train2, X_val2, y_train2, y_val2 = train_test_split(X_temp2, y_temp2, test_size=0.15, random_state=42, stratify=y_temp2)

model_v2 = RandomForestClassifier(n_estimators=200, random_state=42)
model_v2.fit(X_train2, y_train2)
print("New validation accuracy:", accuracy_score(y_val2, model_v2.predict(X_val2)))

New validation accuracy: 0.9840319361277445


In [13]:
import glob

test_images = glob.glob('webcam_test_*.jpg')

correct = 0
total = 0

for img_path in test_images:
    true_label = img_path.split('_')[2]

    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = hands.process(image_rgb)

    if result.multi_hand_landmarks:
        landmarks = result.multi_hand_landmarks[0]
        row = normalize_landmarks(landmarks)

        pred = model_v2.predict([row])[0]
        total += 1
        if pred == true_label:
            correct += 1
        print(f"{img_path}: True={true_label}, Predicted={pred}")

print(f"\nModel v2 Webcam Accuracy: {correct}/{total} = {correct/total:.2%}")

/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

webcam_test_H_16.jpg: True=H, Predicted=H
webcam_test_N_6.jpg: True=N, Predicted=N
webcam_test_N_4.jpg: True=N, Predicted=N
webcam_test_N_5.jpg: True=N, Predicted=N
webcam_test_H_15.jpg: True=H, Predicted=H
webcam_test_M_7.jpg: True=M, Predicted=N
webcam_test_N_12.jpg: True=N, Predicted=M
webcam_test_M_1.jpg: True=M, Predicted=M


/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

webcam_test_M_3.jpg: True=M, Predicted=M
webcam_test_N_10.jpg: True=N, Predicted=N
webcam_test_N_11.jpg: True=N, Predicted=N
webcam_test_M_2.jpg: True=M, Predicted=M
webcam_test_A_2.jpg: True=A, Predicted=A
webcam_test_A_3.jpg: True=A, Predicted=A
webcam_test_G_13.jpg: True=G, Predicted=G
webcam_test_A_1.jpg: True=A, Predicted=A
webcam_test_A_0.jpg: True=A, Predicted=A
webcam_test_M_9.jpg: True=M, Predicted=N
webcam_test_A_4.jpg: True=A, Predicted=N
webcam_test_A_5.jpg: True=A, Predicted=S
webcam_test_M_8.jpg: True=M, Predicted=M
webcam_test_A_6.jpg: True=A, Predicted=W
webcam_test_G_14.jpg: True=G, Predicted=G
webcam_test_B_5.jpg: True=B, Predicted=B
webcam_test_B_4.jpg: True=B, Predicted=B
webcam_test_B_6.jpg: True=B, Predicted=B

Model v2 Webcam Accuracy: 20/26 = 76.92%


/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

### Hyperparameter Tuning

In [15]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 20, 30],
    'min_samples_leaf': [1, 2, 5],
    'max_features': ['sqrt', 'log2']
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    n_jobs=-1,
    verbose=1  # shows progress as it tries combinations
)

grid.fit(X_train2, y_train2)  # using your augmented training data

print("Best parameters:", grid.best_params_)
print("Best cross-validation accuracy:", grid.best_score_)

Fitting 3 folds for each of 54 candidates, totalling 162 fits
Best parameters: {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'n_estimators': 200}
Best cross-validation accuracy: 0.9832276250880904


In [16]:
best_model = grid.best_estimator_

val_preds = best_model.predict(X_val2)
print("Validation Accuracy (tuned):", accuracy_score(y_val2, val_preds))

Validation Accuracy (tuned): 0.9844311377245509


In [17]:
test_images = glob.glob('webcam_test_*.jpg')
correct = 0
total = 0

for img_path in test_images:
    true_label = img_path.split('_')[2]
    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = hands.process(image_rgb)

    if result.multi_hand_landmarks:
        landmarks = result.multi_hand_landmarks[0]
        row = normalize_landmarks(landmarks)
        pred = best_model.predict([row])[0]
        total += 1
        if pred == true_label:
            correct += 1

print(f"Tuned Model Webcam Accuracy: {correct}/{total} = {correct/total:.2%}")

/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

Tuned Model Webcam Accuracy: 21/26 = 80.77%


/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

## CELLS NOT TO RUN

In [21]:

cap = cv2.VideoCapture(0)
count = 0

while True:
    label = input("Enter the letter you're about to sign (or 'quit' to stop): ").upper()
    if label == 'QUIT':
        break

    print(f"Signing '{label}' — press 's' to save, 'n' for next letter")
    while True:
        ret, frame = cap.read()
        cv2.imshow('Capture - press s to save, n for next letter', frame)
        key = cv2.waitKey(1)
        if key == ord('s'):
            filename = f"webcam_train2_{label}_{count}.jpg"
            cv2.imwrite(filename, frame)
            print(f"Saved {filename}")
            count += 1
        elif key == ord('n'):
            break

cap.release()
cv2.destroyAllWindows()

Signing 'M' — press 's' to save, 'n' for next letter
Saved webcam_train2_M_0.jpg
Saved webcam_train2_M_1.jpg
Saved webcam_train2_M_2.jpg
Saved webcam_train2_M_3.jpg
Saved webcam_train2_M_4.jpg
Saved webcam_train2_M_5.jpg
Saved webcam_train2_M_6.jpg
Saved webcam_train2_M_7.jpg
Saved webcam_train2_M_8.jpg
Saved webcam_train2_M_9.jpg
Saved webcam_train2_M_10.jpg
Saved webcam_train2_M_11.jpg
Saved webcam_train2_M_12.jpg
Saved webcam_train2_M_13.jpg
Saved webcam_train2_M_14.jpg
Saved webcam_train2_M_15.jpg
Saved webcam_train2_M_16.jpg
Saved webcam_train2_M_17.jpg
Saved webcam_train2_M_18.jpg
Saved webcam_train2_M_19.jpg
Saved webcam_train2_M_20.jpg
Saved webcam_train2_M_21.jpg
Saved webcam_train2_M_22.jpg
Saved webcam_train2_M_23.jpg
Saved webcam_train2_M_24.jpg
Saved webcam_train2_M_25.jpg
Saved webcam_train2_M_26.jpg
Saved webcam_train2_M_27.jpg
Saved webcam_train2_M_28.jpg
Saved webcam_train2_M_29.jpg
Saved webcam_train2_M_30.jpg
Saved webcam_train2_M_31.jpg
Saved webcam_train2_M_32.jpg


In [22]:
new_data2 = []

train_images2 = glob.glob('webcam_train2_*.jpg')
print(f"Found {len(train_images2)} new images")

for img_path in train_images2:
    true_label = img_path.split('_')[2]

    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = hands.process(image_rgb)

    if result.multi_hand_landmarks:
        landmarks = result.multi_hand_landmarks[0]
        row = list(normalize_landmarks(landmarks))
        row.append(true_label)
        new_data2.append(row)

columns = []
for i in range(21):
    columns.extend([f'x{i}', f'y{i}', f'z{i}'])
columns.append('label')

new_df2 = pd.DataFrame(new_data2, columns=columns)
print(f"Extracted {len(new_df2)} new samples")

# Merge with your previously augmented dataset
df_combined2 = pd.concat([df_combined, new_df2], ignore_index=True)
df_combined2.to_csv('train_landmarks_augmented_v2.csv', index=False)
print(f"Combined dataset: {len(df_combined2)} samples")

Found 124 new images


/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Extracted 119 new samples
Combined dataset: 20988 samples


In [23]:
X_aug2 = df_combined2.drop(columns='label')
y_aug2 = df_combined2['label']

X_temp3, X_test3, y_temp3, y_test3 = train_test_split(X_aug2, y_aug2, test_size=0.2, random_state=42, stratify=y_aug2)
X_train3, X_val3, y_train3, y_val3 = train_test_split(X_temp3, y_temp3, test_size=0.15, random_state=42, stratify=y_temp3)

model_v3 = RandomForestClassifier(n_estimators=200, max_features='log2', min_samples_leaf=1, random_state=42)
model_v3.fit(X_train3, y_train3)
print("Validation Accuracy:", accuracy_score(y_val3, model_v3.predict(X_val3)))

Validation Accuracy: 0.9849146486701071


In [24]:
test_images = glob.glob('webcam_test_*.jpg')
correct = 0
total = 0

for img_path in test_images:
    true_label = img_path.split('_')[2]
    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = hands.process(image_rgb)

    if result.multi_hand_landmarks:
        landmarks = result.multi_hand_landmarks[0]
        row = normalize_landmarks(landmarks)
        pred = model_v3.predict([row])[0]
        total += 1
        if pred == true_label:
            correct += 1

print(f"Model v3 Webcam Accuracy: {correct}/{total} = {correct/total:.2%}")

/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

Model v3 Webcam Accuracy: 20/26 = 76.92%


/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

In [25]:
test_images = glob.glob('webcam_test_*.jpg')

for img_path in test_images:
    true_label = img_path.split('_')[2]
    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = hands.process(image_rgb)
    if result.multi_hand_landmarks:
        landmarks = result.multi_hand_landmarks[0]
        row = normalize_landmarks(landmarks)
        pred = model_v3.predict([row])[0]
        marker = "✓" if pred == true_label else "✗"
        print(f"{marker} {img_path}: True={true_label}, Predicted={pred}")

/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

✓ webcam_test_H_16.jpg: True=H, Predicted=H
✓ webcam_test_N_6.jpg: True=N, Predicted=N
✓ webcam_test_N_4.jpg: True=N, Predicted=N
✓ webcam_test_N_5.jpg: True=N, Predicted=N
✓ webcam_test_H_15.jpg: True=H, Predicted=H
✗ webcam_test_M_7.jpg: True=M, Predicted=N
✓ webcam_test_N_12.jpg: True=N, Predicted=N
✓ webcam_test_M_1.jpg: True=M, Predicted=M
✓ webcam_test_M_3.jpg: True=M, Predicted=M


/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

✓ webcam_test_N_10.jpg: True=N, Predicted=N
✓ webcam_test_N_11.jpg: True=N, Predicted=N
✓ webcam_test_M_2.jpg: True=M, Predicted=M
✓ webcam_test_A_2.jpg: True=A, Predicted=A
✓ webcam_test_A_3.jpg: True=A, Predicted=A
✓ webcam_test_G_13.jpg: True=G, Predicted=G
✓ webcam_test_A_1.jpg: True=A, Predicted=A
✓ webcam_test_A_0.jpg: True=A, Predicted=A
✗ webcam_test_M_9.jpg: True=M, Predicted=N
✗ webcam_test_A_4.jpg: True=A, Predicted=N
✗ webcam_test_A_5.jpg: True=A, Predicted=S
✗ webcam_test_M_8.jpg: True=M, Predicted=N
✗ webcam_test_A_6.jpg: True=A, Predicted=W
✓ webcam_test_G_14.jpg: True=G, Predicted=G
✓ webcam_test_B_5.jpg: True=B, Predicted=B
✓ webcam_test_B_4.jpg: True=B, Predicted=B
✓ webcam_test_B_6.jpg: True=B, Predicted=B


/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

In [26]:

cap = cv2.VideoCapture(0)
count = 0
label = "M"

print(f"Signing '{label}' — press 's' to save, 'q' to quit")
while True:
    ret, frame = cap.read()
    cv2.imshow('Capture - press s to save, q to quit', frame)
    key = cv2.waitKey(1)
    if key == ord('s'):
        filename = f"webcam_train3_M_{count}.jpg"
        cv2.imwrite(filename, frame)
        print(f"Saved {filename}")
        count += 1
    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Signing 'M' — press 's' to save, 'q' to quit
Saved webcam_train3_M_0.jpg
Saved webcam_train3_M_1.jpg
Saved webcam_train3_M_2.jpg
Saved webcam_train3_M_3.jpg
Saved webcam_train3_M_4.jpg
Saved webcam_train3_M_5.jpg
Saved webcam_train3_M_6.jpg
Saved webcam_train3_M_7.jpg
Saved webcam_train3_M_8.jpg
Saved webcam_train3_M_9.jpg
Saved webcam_train3_M_10.jpg
Saved webcam_train3_M_11.jpg
Saved webcam_train3_M_12.jpg
Saved webcam_train3_M_13.jpg
Saved webcam_train3_M_14.jpg
Saved webcam_train3_M_15.jpg
Saved webcam_train3_M_16.jpg
Saved webcam_train3_M_17.jpg
Saved webcam_train3_M_18.jpg
Saved webcam_train3_M_19.jpg
Saved webcam_train3_M_20.jpg
Saved webcam_train3_M_21.jpg
Saved webcam_train3_M_22.jpg
Saved webcam_train3_M_23.jpg
Saved webcam_train3_M_24.jpg
Saved webcam_train3_M_25.jpg
Saved webcam_train3_M_26.jpg
Saved webcam_train3_M_27.jpg
Saved webcam_train3_M_28.jpg
Saved webcam_train3_M_29.jpg
Saved webcam_train3_M_30.jpg
Saved webcam_train3_M_31.jpg
Saved webcam_train3_M_32.jpg
Saved we

In [27]:

new_data3 = []

train_images3 = glob.glob('webcam_train3_*.jpg')
print(f"Found {len(train_images3)} new images")

for img_path in train_images3:
    true_label = img_path.split('_')[2]

    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = hands.process(image_rgb)

    if result.multi_hand_landmarks:
        landmarks = result.multi_hand_landmarks[0]
        row = list(normalize_landmarks(landmarks))
        row.append(true_label)
        new_data3.append(row)

columns = []
for i in range(21):
    columns.extend([f'x{i}', f'y{i}', f'z{i}'])
columns.append('label')

new_df3 = pd.DataFrame(new_data3, columns=columns)
print(f"Extracted {len(new_df3)} new samples")

# Merge with your previous combined dataset (df_combined2 already has original + train + train2)
df_final = pd.concat([df_combined2, new_df3], ignore_index=True)
df_final.to_csv('train_landmarks_final.csv', index=False)
print(f"Final combined dataset: {len(df_final)} samples")

Found 36 new images


/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Extracted 24 new samples
Final combined dataset: 21012 samples


In [28]:
X_final = df_final.drop(columns='label')
y_final = df_final['label']

X_temp4, X_test4, y_temp4, y_test4 = train_test_split(X_final, y_final, test_size=0.2, random_state=42, stratify=y_final)
X_train4, X_val4, y_train4, y_val4 = train_test_split(X_temp4, y_temp4, test_size=0.15, random_state=42, stratify=y_temp4)

model_final = RandomForestClassifier(n_estimators=200, max_features='log2', min_samples_leaf=1, random_state=42)
model_final.fit(X_train4, y_train4)
print("Validation Accuracy:", accuracy_score(y_val4, model_final.predict(X_val4)))

Validation Accuracy: 0.985725614591594


In [29]:
test_images = glob.glob('webcam_test_*.jpg')
correct = 0
total = 0

for img_path in test_images:
    true_label = img_path.split('_')[2]
    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = hands.process(image_rgb)

    if result.multi_hand_landmarks:
        landmarks = result.multi_hand_landmarks[0]
        row = normalize_landmarks(landmarks)
        pred = model_final.predict([row])[0]
        marker = "✓" if pred == true_label else "✗"
        total += 1
        if pred == true_label:
            correct += 1
        print(f"{marker} {img_path}: True={true_label}, Predicted={pred}")

print(f"\nFinal Model Webcam Accuracy: {correct}/{total} = {correct/total:.2%}")

/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

✓ webcam_test_H_16.jpg: True=H, Predicted=H
✓ webcam_test_N_6.jpg: True=N, Predicted=N
✓ webcam_test_N_4.jpg: True=N, Predicted=N
✓ webcam_test_N_5.jpg: True=N, Predicted=N
✓ webcam_test_H_15.jpg: True=H, Predicted=H
✗ webcam_test_M_7.jpg: True=M, Predicted=N
✗ webcam_test_N_12.jpg: True=N, Predicted=M
✓ webcam_test_M_1.jpg: True=M, Predicted=M
✓ webcam_test_M_3.jpg: True=M, Predicted=M


/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

✓ webcam_test_N_10.jpg: True=N, Predicted=N
✓ webcam_test_N_11.jpg: True=N, Predicted=N
✓ webcam_test_M_2.jpg: True=M, Predicted=M
✓ webcam_test_A_2.jpg: True=A, Predicted=A
✓ webcam_test_A_3.jpg: True=A, Predicted=A
✓ webcam_test_G_13.jpg: True=G, Predicted=G
✓ webcam_test_A_1.jpg: True=A, Predicted=A
✓ webcam_test_A_0.jpg: True=A, Predicted=A
✗ webcam_test_M_9.jpg: True=M, Predicted=N
✗ webcam_test_A_4.jpg: True=A, Predicted=N
✗ webcam_test_A_5.jpg: True=A, Predicted=S
✗ webcam_test_M_8.jpg: True=M, Predicted=N
✗ webcam_test_A_6.jpg: True=A, Predicted=W
✓ webcam_test_G_14.jpg: True=G, Predicted=G
✓ webcam_test_B_5.jpg: True=B, Predicted=B
✓ webcam_test_B_4.jpg: True=B, Predicted=B
✓ webcam_test_B_6.jpg: True=B, Predicted=B

Final Model Webcam Accuracy: 19/26 = 73.08%


/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

In [30]:
print("Validation Accuracy:", accuracy_score(y_val2, best_model.predict(X_val2)))

Validation Accuracy: 0.9844311377245509


# Using CNN

In [2]:
#Step 1: Load and preprocess images

import os
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

BASE_PATH = "asl_alphabet_train"
IMG_SIZE = 64  # resize all images to 64x64
IMAGES_PER_CLASS = 1000

classes_to_use = [chr(i) for i in range(65, 91)] + ['space', 'nothing']

X = []
y = []

for label in classes_to_use:
    folder_path = os.path.join(BASE_PATH, label)
    if not os.path.isdir(folder_path):
        print(f"Warning: folder not found for {label}, skipping")
        continue

    img_list = os.listdir(folder_path)[:IMAGES_PER_CLASS]

    for img_name in tqdm(img_list, desc=label):
        img_path = os.path.join(folder_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        X.append(image)
        y.append(label)

X = np.array(X, dtype='float32') / 255.0  # normalize pixel values to 0-1
y = np.array(y)

print(f"Loaded {len(X)} images, shape: {X.shape}")

nothing: 100%|██████████| 1000/1000 [00:00<00:00, 4392.47it/s]


Loaded 28000 images, shape: (28000, 64, 64, 3)


In [3]:
#Step 2: Encode labels and split data

le = LabelEncoder()
y_encoded = le.fit_transform(y)  # converts letters to numbers (A=0, B=1, ...)
num_classes = len(le.classes_)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Train: 19040, Val: 3360, Test: 5600


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Convert your existing X, y_encoded (from Steps 1 & 2) into PyTorch tensors
X_train_t = torch.tensor(X_train).permute(0, 3, 1, 2)  # reorder to (batch, channels, H, W)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t = torch.tensor(X_val).permute(0, 3, 1, 2)
y_val_t = torch.tensor(y_val, dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv(x)
        return self.fc(x)

model_cnn = SimpleCNN(num_classes=num_classes)
optimizer = optim.Adam(model_cnn.parameters())
criterion = nn.CrossEntropyLoss()

for epoch in range(15):
    model_cnn.train()
    total_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model_cnn(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model_cnn.eval()
    with torch.no_grad():
        val_preds = model_cnn(X_val_t)
        val_acc = (val_preds.argmax(1) == y_val_t).float().mean().item()

    print(f"Epoch {epoch+1}: Loss={total_loss:.3f}, Val Accuracy={val_acc:.4f}")


Epoch 1: Loss=1591.735, Val Accuracy=0.5979
Epoch 2: Loss=825.054, Val Accuracy=0.7762
Epoch 3: Loss=577.505, Val Accuracy=0.8670
Epoch 4: Loss=455.091, Val Accuracy=0.9018
Epoch 5: Loss=375.122, Val Accuracy=0.9342
Epoch 6: Loss=321.242, Val Accuracy=0.9423
Epoch 7: Loss=272.373, Val Accuracy=0.9673
Epoch 8: Loss=247.121, Val Accuracy=0.9670
Epoch 9: Loss=226.028, Val Accuracy=0.9699
Epoch 10: Loss=206.941, Val Accuracy=0.9774
Epoch 11: Loss=181.216, Val Accuracy=0.9783
Epoch 12: Loss=172.562, Val Accuracy=0.9583
Epoch 13: Loss=161.807, Val Accuracy=0.9810
Epoch 14: Loss=144.801, Val Accuracy=0.9815
Epoch 15: Loss=140.884, Val Accuracy=0.9804


In [5]:
import glob

def predict_webcam_image(img_path, model, le, img_size=64):
    image = cv2.imread(img_path)
    image = cv2.resize(image, (img_size, img_size))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.astype('float32') / 255.0
    
    img_tensor = torch.tensor(image).permute(2, 0, 1).unsqueeze(0)  # add batch dim
    
    model.eval()
    with torch.no_grad():
        pred = model(img_tensor)
        pred_class = pred.argmax(1).item()
    
    return le.inverse_transform([pred_class])[0]

test_images = glob.glob('webcam_test_*.jpg')
correct = 0
total = 0

for img_path in test_images:
    true_label = img_path.split('_')[2]
    pred_label = predict_webcam_image(img_path, model_cnn, le)
    
    total += 1
    if pred_label == true_label:
        correct += 1
    print(f"{img_path}: True={true_label}, Predicted={pred_label}")

print(f"\nCNN Webcam Accuracy: {correct}/{total} = {correct/total:.2%}")

webcam_test_H_16.jpg: True=H, Predicted=N
webcam_test_N_6.jpg: True=N, Predicted=I
webcam_test_N_4.jpg: True=N, Predicted=N
webcam_test_N_5.jpg: True=N, Predicted=I
webcam_test_H_15.jpg: True=H, Predicted=N
webcam_test_M_7.jpg: True=M, Predicted=L
webcam_test_N_12.jpg: True=N, Predicted=M
webcam_test_M_1.jpg: True=M, Predicted=I
webcam_test_M_3.jpg: True=M, Predicted=I
webcam_test_N_10.jpg: True=N, Predicted=M
webcam_test_N_11.jpg: True=N, Predicted=M
webcam_test_M_2.jpg: True=M, Predicted=I
webcam_test_A_2.jpg: True=A, Predicted=N
webcam_test_A_3.jpg: True=A, Predicted=M
webcam_test_G_13.jpg: True=G, Predicted=N
webcam_test_A_1.jpg: True=A, Predicted=D
webcam_test_A_0.jpg: True=A, Predicted=M
webcam_test_M_9.jpg: True=M, Predicted=L
webcam_test_A_4.jpg: True=A, Predicted=N
webcam_test_A_5.jpg: True=A, Predicted=N
webcam_test_M_8.jpg: True=M, Predicted=M
webcam_test_A_6.jpg: True=A, Predicted=I
webcam_test_G_14.jpg: True=G, Predicted=N
webcam_test_B_5.jpg: True=B, Predicted=R
webcam_te

# Space/Word-Detection

In [18]:
import cv2

cap = cv2.VideoCapture(0)

current_word = ""
full_sentence = ""
last_confirmed_letter = None
stable_letter = None
stable_count = 0
STABILITY_THRESHOLD = 25

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(frame_rgb)

    predicted_letter = None
    if result.multi_hand_landmarks:
        landmarks = result.multi_hand_landmarks[0]
        row = normalize_landmarks(landmarks)
        predicted_letter = best_model.predict([row])[0]

    if predicted_letter != stable_letter:
        stable_letter = predicted_letter
        stable_count = 1
        last_confirmed_letter = None
    else:
        stable_count += 1

    if stable_count == STABILITY_THRESHOLD and predicted_letter != last_confirmed_letter and predicted_letter is not None:
        if predicted_letter == 'space':
            full_sentence += current_word + " "
            current_word = ""
        elif predicted_letter == 'nothing':
            pass
        else:
            current_word += predicted_letter
        last_confirmed_letter = predicted_letter

    if predicted_letter == 'nothing':
        last_confirmed_letter = None

    display_text = f"{full_sentence}{current_word}"
    cv2.putText(frame, display_text, (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Detected: {predicted_letter}", (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
    cv2.putText(frame, "Press 'b' = delete letter, 'q' = quit", (10, 460), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
    cv2.imshow('ASL Fingerspelling', frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('b'):
        if current_word:
            current_word = current_word[:-1]
        elif full_sentence:
            full_sentence = full_sentence.rstrip()

cap.release()
cv2.destroyAllWindows()
print("Final sentence:", full_sentence + current_word)

/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

Final sentence: P


# text-to-speech

In [19]:
import pyttsx3

engine = pyttsx3.init()
engine.setProperty('rate', 150)  # speaking speed, adjust if too fast/slow

cap = cv2.VideoCapture(0)

current_word = ""
full_sentence = ""
last_confirmed_letter = None
stable_letter = None
stable_count = 0
STABILITY_THRESHOLD = 25

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(frame_rgb)

    predicted_letter = None
    if result.multi_hand_landmarks:
        landmarks = result.multi_hand_landmarks[0]
        row = normalize_landmarks(landmarks)
        predicted_letter = best_model.predict([row])[0]

    if predicted_letter != stable_letter:
        stable_letter = predicted_letter
        stable_count = 1
        last_confirmed_letter = None
    else:
        stable_count += 1

    if stable_count == STABILITY_THRESHOLD and predicted_letter != last_confirmed_letter and predicted_letter is not None:
        if predicted_letter == 'space':
            full_sentence += current_word + " "
            current_word = ""
        elif predicted_letter == 'nothing':
            pass
        else:
            current_word += predicted_letter
        last_confirmed_letter = predicted_letter

    if predicted_letter == 'nothing':
        last_confirmed_letter = None

    display_text = f"{full_sentence}{current_word}"
    cv2.putText(frame, display_text, (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Detected: {predicted_letter}", (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
    cv2.putText(frame, "b=delete  s=speak  q=quit", (10, 460), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
    cv2.imshow('ASL Fingerspelling', frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('b'):
        if current_word:
            current_word = current_word[:-1]
        elif full_sentence:
            full_sentence = full_sentence.rstrip()
    elif key == ord('s'):  # speak the sentence so far
        text_to_speak = (full_sentence + current_word).strip()
        if text_to_speak:
            engine.say(text_to_speak)
            engine.runAndWait()

cap.release()
cv2.destroyAllWindows()
print("Final sentence:", full_sentence + current_word)

/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/Users/arshiya/python/demo/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning:

Final sentence: H


In [20]:
import joblib
joblib.dump(best_model, 'best_model.pkl')

['best_model.pkl']